In [ ]:
DATA_DOWNLOADED = True
DFS_PICKLED = True
MODEL_TRAINED = False
BERT_MODEL = 'distilbert-base-uncased'

# Sentiment analysis
Using sample selection based on similarity to train a neural net to predict sentiment.

## Data
The data used for this project come from Amazon reviews ([source](https://nijianmo.github.io/amazon/index.html))

In [ ]:
if not DATA_DOWNLOADED:
    !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Digital_Music_5.json.gz
    !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Video_Games_5.json.gz
    !wget -P ../data http://deepyeti.ucsd.edu/jianmo/amazon/categoryFilesSmall/Arts_Crafts_and_Sewing_5.json.gz

### Data pre processing
- read json.gz file
- Remove 3 star reviews
- map sentiment to star reviews (1, 2 = negative and 4,5 = positive)
- concatenate review title and review text
- select only relevant columns (concat review and sentiment)
- remove duplicates
- pickle dataframe 

In [ ]:
import utils

if not DFS_PICKLED:
    utils.pre_process('../data/Digital_Music_5.json.gz', 'music')
    utils.pre_process('../data/Video_Games_5.json.gz', 'games')
    utils.pre_process('../data/Arts_Crafts_and_Sewing_5.json.gz', 'art')

### Load dataframes from pickle

In [1]:
import pandas as pd

df_music = pd.read_pickle('../data/pickled_dfs/df_music.pkl')  
df_games = pd.read_pickle('../data/pickled_dfs/df_games.pkl')  
df_art = pd.read_pickle('../data/pickled_dfs/df_art.pkl')  

### Compare domains
Concat all reviews into a big string, and compare the strings to find the cosine similarity.

tfidf score of a word w is `tf(w)*idf(w)`  
Where, tf(w) = Number of times the word appears in a document/Total number of words in the document
and idf(w) = Number of documents/Number of documents that contains word w ([source](https://kanoki.org/2018/12/27/text-matching-cosine-similarity/))

In [ ]:
def make_big_string(df, n):
    '''
    input: dataframe with reviews and number of reviews to compare
    output: all values in the column concatenated
    '''
    # n is the number of reviews we want to compare
    big_string = ' '.join(df.iloc[:n,0].astype(str))
    return big_string.lower()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cos_sim_df(str1, str2):
    '''
    input: 2 strings
    output: cosine similarity and dataframe with tfidf score for each word
    '''
    corpus = [str1, str2]

    # tokanise -> remove strop words, select only words (ignore punctuation, digits, etc)
    vectorizer = TfidfVectorizer(stop_words='english', token_pattern='[a-z]\w+')
    trsfm = vectorizer.fit_transform(corpus)

    return cosine_similarity(trsfm[0:1], trsfm)[0][1], pd.DataFrame(trsfm.toarray(),columns=vectorizer.get_feature_names_out(),index=['str1','str2'])

In [ ]:
len(df_music), len(df_games), len(df_art)

In [ ]:
cs_music_games, df_music_games = cos_sim_df(make_big_string(df_music, 95621), make_big_string(df_games, 364933))
cs_music_games

In [ ]:
cs_music_art, df_music_art = cos_sim_df(make_big_string(df_music, 95621), make_big_string(df_art, 339610))
cs_music_art

## Model training

Train CNN for sentiment analysis

### Tokenize the reviews
Tokenize the reviews to get the embeddings to feed the neural net

### Split datasets
Into train, val and test sets.

In [2]:
from sklearn.model_selection import train_test_split

def split_dataset(df, random_state=42):
    X, y = df['rev_sum'], df['sentiment']

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, train_size = 0.8, stratify=y, random_state=42)
    X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, train_size = 0.5, stratify=y_temp, random_state=42)

    return X_train, y_train, X_dev, y_dev, X_test, y_test

In [3]:
X_train, y_train, X_dev, y_dev, X_test, y_test = split_dataset(df_music)

In [ ]:
# from https://medium.com/pytorch/working-on-natural-language-processing-nlp-with-pytorch-8090c879aadc

# def tokenize(s):
#     return s.split(' ')
# TEXT = data.Field(tokenize=tokenize)
# LABEL = data.LabelField()

In [ ]:
# # from: https://machinelearningmastery.com/prepare-text-data-machine-learning-scikit-learn/
# # not sure what this is doing

# from sklearn.feature_extraction.text import TfidfVectorizer
# # list of text documents
# l = list(X_train.astype(str))
# text = list(map(str.lower, l))

# # create the transform
# vectorizer = TfidfVectorizer(stop_words='english', token_pattern='[a-z]\w+')

# # tokenize and build vocab
# vectorizer.fit(text)

# # summarize
# # print(vectorizer.vocabulary_)
# # print(vectorizer.idf_)

# # encode document
# vector = vectorizer.transform([text[0]])

# # summarize encoded vector
# # print(vector.shape)
# # print(vector.toarray())

In [ ]:
# use this to make CNN https://towardsdatascience.com/sentiment-classification-using-cnn-in-pytorch-fba3c6840430
# from lecture CNN https://github.com/cezannec/CNN_Text_Classification

# BERT

In [16]:
import bert
import torch

if torch.cuda.is_available():       
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


In [ ]:
# # Load the BERT tokenizer
# from transformers import DistilBertTokenizer
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased', do_lower_case=True)

In [ ]:
# Encode our concatenated data
# encoded_ = [tokenizer.encode(sent, add_special_tokens=True) for sent in X_train] # this seems redundant?

## Pre-processing
Getting inputs for the model and masks for train and validation sets.

In [17]:
TEST = True

if TEST:
    TRAIN_SIZE = 1000
    DEV_SIZE = 100

    X_train = X_train[:TRAIN_SIZE]
    y_train = y_train[:TRAIN_SIZE]
    
    X_dev = X_dev[:DEV_SIZE]
    y_dev = y_dev[:DEV_SIZE]

In [18]:
train_inputs, train_masks = bert.preprocessing_for_bert(X_train)

In [19]:
val_inputs, val_masks = bert.preprocessing_for_bert(X_dev)

In [20]:
# Convert other data types to torch.Tensor
train_labels = torch.tensor(y_train.to_numpy())
val_labels = torch.tensor(y_dev.to_numpy())

In [21]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Set batch size. 2 is about the highest that will run on a laptop for testing. 16 or 32 might work on HPC?
BATCH_SIZE = 2

# Create the DataLoader for our training set
train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

# Create the DataLoader for our validation set
val_data = TensorDataset(val_inputs, val_masks, val_labels)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=BATCH_SIZE)

# set seed for reproducibility
bert.set_seed(42)

In [22]:
# initialise model
bert_classifier, optimizer, scheduler = bert.initialize_model2(int(len(train_dataloader)), epochs=2)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_projector.bias', 'vocab_transform.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/sabrina/miniconda3/envs/torch/lib/python3.9/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use thePyTorch implementation torch.optim.AdamW instead, or set `no_depr

In [28]:
# use to reload if making changes to the imported script 
# without needing to restart the kernel
import importlib
importlib.reload(bert)

<module 'bert' from '/mnt/c/Users/sabrina/Documents/GitHub/sentiment-analysis/src/bert.py'>

In [29]:
# train model
bert.train(bert_classifier, optimizer, scheduler, train_dataloader, val_dataloader, epochs=2, evaluation=True)

100%|██████████| 500/500 [01:59<00:00,  4.18it/s]


In [32]:
import pickle
MODEL_DIR = '../artifacts/models/'
FILE_NAME = 'baseline_1000_music'
# pickle.dump(regressor, open('model.pkl','wb'))
pickle.dump(bert_classifier, open('{}.pkl'.format(FILE_NAME), 'wb'))

PicklingError: Can't pickle <class 'bert.BertClassifier'>: it's not the same object as bert.BertClassifier